In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

load_dotenv()

# --- Setup clients ---

if os.getenv("USE_AZURE_CLI_CREDENTIALS", "false").lower() == "true":
    credential = AzureCliCredential()
else:
    credential = DefaultAzureCredential()
    
project_client = AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=credential,
)
openai_client = project_client.get_openai_client()

# --- Step 1: Create an agent ---
agent = project_client.agents.create_version(
    agent_name="TravelBuddy",
    definition=PromptAgentDefinition(
        model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        instructions=(
            "You are TravelBuddy, a friendly travel assistant. "
            "You help users plan trips, suggest destinations, and answer travel questions. "
            "Keep your answers short, concise, and easy to grasp."
        ),
    ),
)
print(f"Agent created — name: {agent.name}, version: {agent.version}")


Agent created — name: TravelBuddy, version: 1


In [3]:
# --- Step 2: Start a conversation ---
conversation = openai_client.conversations.create()
print(f"Conversation started — id: {conversation.id}")

# --- Step 3: First turn ---
user_input = "I have a week off in March. Suggest 3 warm destinations."
print(f"\n********\nUser: {user_input}")
response = openai_client.responses.create(
    conversation=conversation.id,
    input=user_input,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)
print(f"Agent: {response.output_text}")

# --- Step 4: Follow-up (multi-turn) ---
user_input = "Tell me more about the second option. What should I pack?"
print(f"\n********\nUser: {user_input}")
response = openai_client.responses.create(
    conversation=conversation.id,
    input=user_input,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)
print(f"Agent: {response.output_text}")

# --- Cleanup ---
openai_client.conversations.delete(conversation_id=conversation.id)
project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print("Cleaned up.")

Conversation started — id: conv_959063d5f3a1443400lryxiMD3sVpWz5aEsne0QQBz2k10ok3P

********
User: I have a week off in March. Suggest 3 warm destinations.
Agent: - **Canary Islands (Spain)** — reliably mild/warm in March, beaches + volcanic hikes; easy week-long island hop (Tenerife/Lanzarote).  
- **Dubai + Abu Dhabi (UAE)** — hot, sunny, great beaches, desert day trips, lots to do in a week.  
- **Cancún / Riviera Maya (Mexico)** — warm Caribbean weather, swim/snorkel, cenotes, and Mayan ruins (Tulum/Chichén Itzá).

If you tell me your departure region (US/EU/UK, etc.) and budget, I can tailor these to the easiest flights and best-value spots.

********
User: Tell me more about the second option. What should I pack?
Agent: **Dubai + Abu Dhabi in March (1 week)**  
- **Weather:** typically warm-to-hot (around 22–32°C / 72–90°F), sunny, low rain; evenings can feel cooler than midday.  
- **What to do (easy week plan):**  
  - **Dubai (4–5 days):** Burj Khalifa + Dubai Mall fountains, 